# Reconstruction impact: a continuous Raw/SVC workbench

This notebook is a readable, stage-by-stage analysis of one declared Raw/SVC
sample. It keeps the scientific path continuous:

1. declare and load the sample;
2. inspect Raw and SVC independently;
3. partition each side on its own expression matrix;
4. optionally compare membership on a Raw-defined cohort;
5. retain the full Raw Level1 anatomy context;
6. put both sides on a common physical window grid while keeping units and
   rarefaction draws independent;
7. classify SVC State and cross-side Gain separately;
8. calculate native-coordinate Moran I independently;
9. score the optional AUCell demo program;
10. record observations, unavailable stages, and interpretation boundaries.

The notebook calls the same public analysis and method functions used by the
package. It does not call the integrated workflow entrypoint or hide the
analysis in one run cell. Its tables and figures are exploratory evidence
under the requested output directory; the batch runner and static report own
formal published results.

## Input and parameter contract

The default SAMPLE_YAML value is the relative path
../data/example/sample.yaml. Set SAMPLE_YAML to a real sample declaration
before running the notebook on real data. The repository script
scripts/create_example.py creates the synthetic fixture; its demo program is
G0 through G49.

The sample declaration must state the upstream expression scale. The notebook
does not infer normalization, reverse a transformation, or modify either H5AD.
Raw and SVC remain independent objects unless the optional membership stage
explicitly defines a Raw-cohort comparison.

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd

# Make execution robust when Jupyter starts in either the repository root or
# notebooks/. The fallback paths below remain visible and user-editable.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "revise_analysis").is_dir():
        sys.path.insert(0, str(candidate))
        break

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = print

from revise_analysis.io import load_sample
from revise_analysis.analyses import pathway_activity as pathway_workflow
from revise_analysis.analyses import reconstruction_impact as impact
from revise_analysis.analyses import spatial_autocorrelation as moran_workflow
from revise_analysis.methods.partition import compare_membership
from revise_analysis.methods.regions import (
    assign_anatomy_candidates,
    assign_square_windows,
    flag_region_windows,
    select_region_threshold,
)

# Visible inputs. SAMPLE_YAML has the requested environment override and
# ../data/example/sample.yaml fallback.
SAMPLE_YAML = Path(os.environ.get("SAMPLE_YAML", "../data/example/sample.yaml"))
if not SAMPLE_YAML.exists() and Path("data/example/sample.yaml").exists():
    SAMPLE_YAML = Path("data/example/sample.yaml")

OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "../output/notebook/example"))
if not os.environ.get("OUTPUT_DIR") and not Path("../data").exists() and Path("data").exists():
    OUTPUT_DIR = Path("output/notebook/example")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SCOPES = ("All", "Fibroblast", "Mono/Macro", "T")
PARTITION_RESOLUTION = float(os.environ.get("PARTITION_RESOLUTION", "0.5"))
PARTITION_N_TOP_GENES = int(os.environ.get("PARTITION_N_TOP_GENES", "2000"))
SAMPLE_N_UNITS = os.environ.get("SAMPLE_N_UNITS")
PARTITION_SAMPLE_N_UNITS = int(SAMPLE_N_UNITS) if SAMPLE_N_UNITS else None
RAW_PARTITION_SEED = int(os.environ.get("RAW_PARTITION_SEED", "42"))
SVC_PARTITION_SEED = int(os.environ.get("SVC_PARTITION_SEED", "43"))

RUN_MEMBERSHIP = os.environ.get("RUN_MEMBERSHIP", "0").lower() not in {"0", "false", "no"}
MEMBERSHIP_SEED = int(os.environ.get("MEMBERSHIP_SEED", "44"))

WINDOW_SIDE_MICRONS = float(os.environ.get("WINDOW_SIDE_MICRONS", "32.0"))
MIN_WINDOW_UNITS = int(os.environ.get("MIN_WINDOW_UNITS", "4"))
N_WINDOW_DRAWS = int(os.environ.get("N_WINDOW_DRAWS", "200"))
WINDOW_SEEDS = {"raw": 42, "svc": 43}
REGION_N_BOOTSTRAP = int(os.environ.get("REGION_N_BOOTSTRAP", "500"))
ANATOMY_TUMOR_LABEL = os.environ.get("ANATOMY_TUMOR_LABEL", "Tumor")
ANATOMY_NORMAL_LABEL = os.environ.get("ANATOMY_NORMAL_LABEL", "Intestinal Epithelial")
MORAN_N_NEIGHBORS = int(os.environ.get("MORAN_N_NEIGHBORS", "6"))
PATHWAY_AUC_THRESHOLD = float(os.environ.get("PATHWAY_AUC_THRESHOLD", "0.05"))

# The synthetic fixture intentionally contains G0..G599. The first 50 genes
# form a small explicit program. A declared GMT can replace it through
# GENESET_PATH, with optional comma-separated GENESET_NAMES.
GENE_SETS = {"DEMO_PROGRAM": [f"G{i}" for i in range(50)]}
GENESET_PATH = os.environ.get("GENESET_PATH")
GENESET_NAMES = [name for name in os.environ.get("GENESET_NAMES", "").split(",") if name] or None
GENE_SET_SOURCE = "demo G0..G49"
GENE_SET_ERROR = ""
if GENESET_PATH:
    try:
        from revise_analysis.io import read_gene_sets
        GENE_SETS = read_gene_sets(Path(GENESET_PATH), names=GENESET_NAMES)
        GENE_SET_SOURCE = str(GENESET_PATH)
    except (FileNotFoundError, OSError, ValueError) as exc:
        GENE_SETS = {}
        GENE_SET_ERROR = f"{type(exc).__name__}: {exc}"

stage_status = {}
def note(stage, status, reason="", detail=""):
    stage_status[stage] = {"status": status, "reason": reason, "detail": detail}

def slug(value):
    return "".join(char if char.isalnum() else "_" for char in str(value)).strip("_") or "stage"

def save_table(frame, relative_name):
    path = TABLE_DIR / relative_name
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path)
    return path

display(pd.DataFrame([
    {"parameter": "SAMPLE_YAML", "value": str(SAMPLE_YAML)},
    {"parameter": "OUTPUT_DIR", "value": str(OUTPUT_DIR)},
    {"parameter": "SCOPES", "value": ", ".join(SCOPES)},
    {"parameter": "partition", "value": f"resolution={PARTITION_RESOLUTION}, n_top_genes={PARTITION_N_TOP_GENES}"},
    {"parameter": "window", "value": f"{WINDOW_SIDE_MICRONS} microns, min_units={MIN_WINDOW_UNITS}, draws={N_WINDOW_DRAWS}"},
    {"parameter": "region bootstrap", "value": REGION_N_BOOTSTRAP},
    {"parameter": "Moran", "value": f"native k={MORAN_N_NEIGHBORS}"},
    {"parameter": "AUCell", "value": f"threshold={PATHWAY_AUC_THRESHOLD}, source={GENE_SET_SOURCE}"},
]))

## Load the declared sample

Loading validates the YAML contract and reads Raw and SVC without pairing,
normalization, annotation, or input mutation. A missing sample is an input
blocker; later optional stages report their own availability.

In [ ]:
sample = None
input_error = ""
try:
    sample = load_sample(SAMPLE_YAML)
    note("input", "available", detail=f"sample_id={sample.sample_id}")
    print(f"Loaded {sample.sample_id!r} from {sample.source}")
except Exception as exc:
    input_error = f"{type(exc).__name__}: {exc}"
    note("input", "unavailable", input_error)
    display(Markdown(f"**Input unavailable:** {input_error}"))

## Inspect the input contract and axes

The expression scale is a declared upstream fact. Level1 is the broad anatomy
field used by this notebook. Level2 is reported for transparency only; its
absence must not block the full Raw Level1 anatomy stage.

In [ ]:
if sample is None:
    display(Markdown("The input contract cannot be inspected until SAMPLE_YAML loads."))
else:
    level2_available = sample.subtype_key in sample.raw.obs.columns
    level2_reason = "" if level2_available else f"{sample.subtype_key!r} is absent on Raw; no Level2-dependent baseline is requested."
    note("level2_context", "available" if level2_available else "unavailable", level2_reason)
    contract = pd.DataFrame([
        {"field": "sample_id", "value": sample.sample_id},
        {"field": "expression.scale", "value": sample.config.get("expression", {}).get("scale")},
        {"field": "Raw units / genes", "value": f"{sample.raw.n_obs} / {sample.raw.n_vars}"},
        {"field": "SVC units / genes", "value": f"{sample.svc.n_obs} / {sample.svc.n_vars}"},
        {"field": "broad label key", "value": sample.broad_key},
        {"field": "subtype key", "value": sample.subtype_key},
        {"field": "spatial key", "value": sample.spatial_key},
        {"field": "Raw Level2 present", "value": level2_available},
    ])
    display(contract)

## Independent Raw and SVC objects

Sampling is optional and is performed separately for each side. The selected
objects retain their native unit IDs, coordinates, and expression values.
Nothing in this cell asks the two sides to have the same observations.

In [ ]:
side_inputs = {}
side_overview = []

if sample is None:
    note("side_overview", "unavailable", "sample input is unavailable")
else:
    for side, seed in (("raw", RAW_PARTITION_SEED), ("svc", SVC_PARTITION_SEED)):
        try:
            adata, labels = impact.prepare_side(
                sample, side, sample_n_units=PARTITION_SAMPLE_N_UNITS, random_state=seed
            )
            side_inputs[side] = (adata, labels)
            side_overview.append({
                "side": side,
                "n_units": int(adata.n_obs),
                "n_genes": int(adata.n_vars),
                "n_label_values": int(labels.nunique()),
                "sample_seed": seed,
            })
            note(f"side:{side}", "available", detail=f"{adata.n_obs} native units in this notebook view")
        except Exception as exc:
            note(f"side:{side}", "unavailable", f"{type(exc).__name__}: {exc}")

side_overview_table = pd.DataFrame(side_overview)
if not side_overview_table.empty:
    save_table(side_overview_table, "overview.csv")
    display(side_overview_table)

## Independent partitions

Each side is normalized and logged on a working copy inside the partition
method, then reduced, graphed, and Leiden-partitioned with the visible
resolution, feature cap, and seed. The returned labels belong only to that
side and scope. A missing broad scope is recorded as unavailable rather than
reinterpreted.

In [ ]:
partition_results = {}
partition_labels = {}
partition_rows = []

if sample is None:
    note("partitions", "unavailable", "sample input is unavailable")
else:
    for side, (adata, labels) in side_inputs.items():
        seed = RAW_PARTITION_SEED if side == "raw" else SVC_PARTITION_SEED
        for scope in SCOPES:
            stage = f"partition:{side}:{scope}"
            if scope != "All" and not (labels == scope).any():
                note(stage, "unavailable", f"no {scope!r} labels on {side}")
                continue
            try:
                result = impact.partition_scope(
                    adata,
                    labels,
                    scope,
                    resolution=PARTITION_RESOLUTION,
                    n_top_genes=PARTITION_N_TOP_GENES,
                    random_state=seed,
                )
                memberships = result["labels"].rename("partition")
                partition_results[(side, scope)] = result
                partition_labels[(side, scope)] = memberships
                table = memberships.to_frame()
                table["scope"] = scope
                table["side"] = side
                save_table(table, f"partitions_{side}_{slug(scope)}.csv")
                partition_rows.append({"side": side, "scope": scope, **result["summary"]})
                note(stage, "available", detail=f"{result['summary']['n_clusters']} clusters")
            except Exception as exc:
                note(stage, "unavailable", f"{type(exc).__name__}: {exc}")

partition_summary = pd.DataFrame(partition_rows)
if not partition_summary.empty:
    save_table(partition_summary, "partition_summary.csv")
    display(partition_summary)
    first_key = next(iter(partition_labels), None)
    if first_key is not None:
        display(Markdown(f"Example visible labels: {first_key}"))
        display(partition_labels[first_key].head(10).to_frame())

if plt is not None and not partition_summary.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    partition_summary.assign(label=lambda frame: frame["side"] + " / " + frame["scope"]).plot(
        x="label", y="n_clusters", kind="bar", ax=ax, color="#4C78A8"
    )
    ax.set_ylabel("Number of Leiden clusters")
    ax.set_xlabel("Independent side and scope")
    ax.set_title("Partition sizes; labels are native to each side")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "partition_sizes.png", dpi=150)
    plt.show()

## Optional Raw-defined membership comparison

Membership is the only paired component. For each selected Raw scope, the Raw
partition defines the cohort. The SVC partition is recomputed on the
intersection of those explicit unit IDs, then Hungarian matching describes
that comparison edge. This table does not redefine the native SVC population
and is not a prerequisite for Moran, AUCell, or local diversity.

In [ ]:
membership_results = {}
membership_rows = []

if not RUN_MEMBERSHIP:
    note("membership", "disabled", "RUN_MEMBERSHIP is false")
elif sample is None:
    note("membership", "unavailable", "sample input is unavailable")
elif "svc" not in side_inputs:
    note("membership", "unavailable", "SVC side is unavailable")
else:
    # Membership is the one exception to independent side sampling: its
    # Raw-defined IDs are intersected with the full sample.svc object.
    svc_adata = sample.svc
    for scope in SCOPES:
        stage = f"membership:{scope}"
        raw_labels = partition_labels.get(("raw", scope))
        if raw_labels is None:
            note(stage, "unavailable", "Raw parent partition is unavailable")
            continue
        common = raw_labels.index[raw_labels.index.isin(svc_adata.obs_names)]
        if len(common) < 3:
            note(stage, "unavailable", f"only {len(common)} common unit IDs")
            continue
        try:
            svc_partition = impact.compute_partitions(
                svc_adata[common],
                resolution=PARTITION_RESOLUTION,
                n_top_genes=PARTITION_N_TOP_GENES,
                random_state=MEMBERSHIP_SEED,
            )
            comparison = compare_membership(raw_labels.loc[common], svc_partition["labels"])
            membership_results[scope] = comparison
            save_table(comparison.summary, f"membership_summary_{slug(scope)}.csv")
            save_table(comparison.mapping, f"membership_mapping_{slug(scope)}.csv")
            save_table(comparison.contingency, f"membership_contingency_{slug(scope)}.csv")
            save_table(comparison.assignments, f"membership_{slug(scope)}.csv")
            summary_row = comparison.summary.iloc[0].to_dict()
            membership_rows.append({"scope": scope, **summary_row})
            note(stage, "available", detail=f"n_units={summary_row['n_units']}")
        except Exception as exc:
            note(stage, "unavailable", f"{type(exc).__name__}: {exc}")

membership_summary = pd.DataFrame(membership_rows)
if not membership_summary.empty:
    display(membership_summary)
    if plt is not None:
        fig, ax = plt.subplots(figsize=(8, 4))
        membership_summary.assign(label=lambda frame: frame["scope"]).plot(
            x="label", y="st_unit_change_fraction", kind="bar", ax=ax, color="#F58518"
        )
        ax.set_ylabel("Matched unit change fraction")
        ax.set_xlabel("Raw-defined scope")
        ax.set_title("Raw-defined membership comparison")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "membership_change.png", dpi=150)
        plt.show()

## Full Raw Level1 anatomy context

Anatomy is built from every Raw unit and its declared broad label. It is
assigned to the same physical window grid used later for local metrics. No
Level2 mapping, selected local subset, partition label, or SVC annotation is
used to invent anatomy. Tumor, Normal, Interface, and Other are descriptive
candidates controlled by the visible labels below.

In [ ]:
anatomy_context = pd.DataFrame()
anatomy_windows = pd.DataFrame()
physical_origin = None
window_side_coordinate = None

if sample is None:
    note("anatomy", "unavailable", "sample input is unavailable")
else:
    try:
        anatomy_context = impact.raw_anatomy_context(sample)
        scale = float(sample.config.get("spatial", {}).get("microns_per_coordinate"))
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError("spatial.microns_per_coordinate must be positive")
        physical_origin = (
            float(anatomy_context["x"].min()),
            float(anatomy_context["y"].min()),
        )
        window_side_coordinate = WINDOW_SIDE_MICRONS / scale
        assignments = assign_square_windows(
            anatomy_context[["x", "y"]],
            window_side_length=window_side_coordinate,
            origin=physical_origin,
        )
        anatomy_windows = assign_anatomy_candidates(
            assignments,
            anatomy_context["broad_label"],
            tumor_label=ANATOMY_TUMOR_LABEL,
            normal_source_label=ANATOMY_NORMAL_LABEL,
        )
        anatomy_context = (
            anatomy_context.join(assignments[["window_id"]])
            .join(anatomy_windows.set_index("window_id")[["level1_region"]], on="window_id")
        )
        save_table(anatomy_context, "raw_anatomy_context.csv")
        save_table(anatomy_windows, "raw_anatomy_windows.csv")
        note("anatomy", "available", detail=f"full Raw units={len(anatomy_context)}")
        display(pd.DataFrame([
            {"parameter": "window side (microns)", "value": WINDOW_SIDE_MICRONS},
            {"parameter": "microns per coordinate", "value": scale},
            {"parameter": "window side (coordinates)", "value": window_side_coordinate},
            {"parameter": "origin", "value": physical_origin},
            {"parameter": "anatomy source", "value": "full Raw Level1"},
        ]))
        display(anatomy_windows.head(12))
    except Exception as exc:
        note("anatomy", "unavailable", f"{type(exc).__name__}: {exc}")

if plt is not None and not anatomy_context.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    colors = {"Tumor": "#E45756", "Normal": "#54A24B", "Interface": "#B279A2", "Other": "#9D9D9D"}
    for region, frame in anatomy_context.groupby("level1_region", dropna=False):
        ax.scatter(frame["x"], frame["y"], s=12, alpha=0.65, label=str(region), color=colors.get(str(region), "#9D9D9D"))
    ax.set_title("Full Raw Level1 anatomy context")
    ax.set_xlabel("native x coordinate")
    ax.set_ylabel("native y coordinate")
    ax.legend(title="window anatomy", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "raw_anatomy_context.png", dpi=150)
    plt.show()

## Common physical windows with independent local draws

The full Raw origin and the configured microns-per-coordinate scale define one
common square grid. Each side contributes its own native units to that grid.
For every eligible window, the method independently draws
MIN_WINDOW_UNITS units N_WINDOW_DRAWS times and reports:

- H = minus sum of p_k log(p_k), Shannon entropy;
- N_eff = exp(H), effective observed diversity;
- evenness = N_eff / K_obs.

A window below the support threshold remains visible with coordinates and unit
count, but its diversity values are unavailable. Raw and SVC use separate
random seeds so the rarefaction draws are not reused.

In [ ]:
window_tables = {}

if sample is None:
    note("windows", "unavailable", "sample input is unavailable")
elif physical_origin is None or window_side_coordinate is None:
    note("windows", "unavailable", "common physical geometry is unavailable")
else:
    for (side, scope), labels in sorted(partition_labels.items()):
        stage = f"windows:{side}:{scope}"
        if side not in side_inputs:
            note(stage, "unavailable", f"{side} side is unavailable")
            continue
        adata, _ = side_inputs[side]
        try:
            table = impact.native_window_diversity(
                sample,
                side,
                labels,
                adata=adata,
                window_side_length=window_side_coordinate,
                origin=physical_origin,
                min_units=MIN_WINDOW_UNITS,
                n_draws=N_WINDOW_DRAWS,
                random_state=WINDOW_SEEDS[side],
            )
            window_tables[(side, scope)] = table
            save_table(table, f"window_diversity_{side}_{slug(scope)}.csv")
            note(stage, "available", detail=f"{len(table)} common-grid windows")
        except Exception as exc:
            note(stage, "unavailable", f"{type(exc).__name__}: {exc}")

# If Raw Level2 is supplied upstream, calculate its native baseline with
# the same window method. It is descriptive context and never feeds
# State or Gain region selection. SVC Level2 is not required.
raw_level2_baselines = {}
raw_level2_comparisons = {}
if sample is not None and physical_origin is not None and window_side_coordinate is not None:
    if sample.subtype_key not in sample.raw.obs.columns:
        note("raw_level2_baseline", "unavailable", f"Raw column {sample.subtype_key!r} is absent")
    else:
        raw_adata, raw_broad_labels = side_inputs.get("raw", (None, None))
        raw_level2_labels = sample.labels("raw", sample.subtype_key)
        if raw_adata is None or raw_level2_labels.isna().any():
            note("raw_level2_baseline", "unavailable", "Raw Level2 labels are missing in the notebook view")
        else:
            for scope in SCOPES:
                stage = f"raw_level2_baseline:{scope}"
                selected = raw_adata.obs_names if scope == "All" else raw_broad_labels.index[raw_broad_labels == scope]
                if len(selected) == 0:
                    note(stage, "unavailable", f"no Raw {scope!r} units")
                    continue
                try:
                    baseline = impact.native_window_diversity(
                        sample, "raw", raw_level2_labels.reindex(selected), adata=raw_adata[selected],
                        window_side_length=window_side_coordinate, origin=physical_origin,
                        min_units=MIN_WINDOW_UNITS, n_draws=N_WINDOW_DRAWS,
                        random_state=RAW_PARTITION_SEED + 17,
                    )
                    raw_level2_baselines[scope] = baseline
                    save_table(baseline, f"window_diversity_raw_level2_baseline_{slug(scope)}.csv")
                    svc_parent = window_tables.get(("svc", scope))
                    if svc_parent is None:
                        note(stage, "available", "baseline saved; SVC parent table unavailable")
                        continue
                    raw_indexed = baseline.set_index("window_id")
                    svc_indexed = svc_parent.set_index("window_id")
                    common = raw_indexed.index.intersection(svc_indexed.index)
                    valid = raw_indexed.loc[common, "valid_window"].astype(bool) & svc_indexed.loc[common, "valid_window"].astype(bool)
                    if not valid.any():
                        note(stage, "available", "baseline saved; no common valid windows for descriptive comparison")
                        continue
                    comparison = pd.DataFrame({
                        "raw_level2_baseline_neff": raw_indexed.loc[common, "neff"],
                        "svc_parent_leiden_neff": svc_indexed.loc[common, "neff"],
                        "window_x": raw_indexed.loc[common, "window_x"],
                        "window_y": raw_indexed.loc[common, "window_y"],
                    }).loc[valid]
                    comparison["descriptive_delta_neff"] = comparison["svc_parent_leiden_neff"] - comparison["raw_level2_baseline_neff"]
                    comparison = comparison.reset_index()
                    comparison["valid_window"] = True
                    raw_level2_comparisons[scope] = comparison
                    save_table(comparison, f"raw_level2_baseline_vs_svc_{slug(scope)}_common_valid_windows.csv")
                    note(stage, "available", detail=f"comparison windows={len(comparison)}")
                except Exception as exc:
                    note(stage, "unavailable", f"{type(exc).__name__}: {exc}")

if raw_level2_baselines:
    display(pd.DataFrame([
        {"scope": scope, "n_windows": len(table), "n_valid_windows": int(table["valid_window"].sum()), "comparison_windows": len(raw_level2_comparisons.get(scope, []))}
        for scope, table in raw_level2_baselines.items()
    ]))

window_overview = []
for (side, scope), table in window_tables.items():
    window_overview.append({
        "side": side,
        "scope": scope,
        "n_windows": len(table),
        "n_valid_windows": int(table["valid_window"].sum()),
        "n_units": int(table["n_units"].sum()),
        "min_units": MIN_WINDOW_UNITS,
        "draws": N_WINDOW_DRAWS,
        "seed": WINDOW_SEEDS[side],
    })
window_overview = pd.DataFrame(window_overview)
if not window_overview.empty:
    save_table(window_overview, "window_overview.csv")
    display(window_overview)

if plt is not None:
    for side in ("raw", "svc"):
        table = window_tables.get((side, "All"))
        if table is None or table.empty:
            continue
        fig, ax = plt.subplots(figsize=(7, 5))
        values = table["neff"].where(table["valid_window"].astype(bool))
        plot = ax.scatter(table["window_x"], table["window_y"], c=values, s=70, cmap="viridis")
        fig.colorbar(plot, ax=ax, label="N_eff (valid windows)")
        ax.set_title(f"{side.upper()} native local diversity on common physical grid")
        ax.set_xlabel("window x")
        ax.set_ylabel("window y")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"window_neff_{side}.png", dpi=150)
        plt.show()

## State and Gain regions

State and Gain answer different questions.

- State uses native SVC local diversity for each declared parent scope. It
  does not compare Raw and SVC and does not require shared unit IDs.
- Gain uses the SVC minus Raw N_eff difference in common valid windows for the
  same scope. It is computed only after both side-specific tables exist.

The region threshold is accepted only when the survival-curve breakpoint is
stable across the visible bootstrap count. If support or stability is
insufficient, the threshold remains unavailable; no percentile or map-derived
replacement is used. A State or Gain region is descriptive evidence and is
not itself a claim of biological improvement.

In [ ]:
state_tables = {}
state_thresholds = {}
state_region_tables = {}
gain_tables = {}
gain_thresholds = {}
gain_region_tables = {}
region_rows = []

if sample is None:
    note("regions", "unavailable", "sample input is unavailable")
else:
    # State is defined for SVC parent scopes; All is intentionally not treated
    # as a parent-specific State region.
    for scope in SCOPES:
        if scope == "All":
            continue
        stage = f"state:{scope}"
        svc_table = window_tables.get(("svc", scope))
        if svc_table is None:
            note(stage, "unavailable", "SVC parent window table is unavailable")
            continue
        state_tables[scope] = svc_table
        valid_values = svc_table.loc[svc_table["valid_window"].astype(bool), "neff"]
        threshold, audit = select_region_threshold(
            valid_values,
            n_bootstrap=REGION_N_BOOTSTRAP,
            random_state=WINDOW_SEEDS["svc"],
        )
        state_thresholds[scope] = threshold
        save_table(audit, f"state_{slug(scope)}_threshold_bootstrap.csv") if not audit.empty else None
        (TABLE_DIR / f"state_{slug(scope)}_threshold.json").write_text(
            json.dumps(threshold, indent=2, default=float) + "\n",
            encoding="utf-8",
        )
        if threshold.get("threshold") is None:
            state_region_tables[scope] = pd.DataFrame()
            note(stage, "unavailable", f"threshold status={threshold.get('status')}")
            region_rows.append({"type": "State", "scope": scope, "status": threshold.get("status"), "threshold": np.nan})
            continue
        summary, flagged = flag_region_windows(
            svc_table,
            threshold=float(threshold["threshold"]),
            value_column="neff",
            window_side_length=WINDOW_SIDE_MICRONS,
        )
        state_region_tables[scope] = flagged
        save_table(flagged, f"state_{slug(scope)}_region_windows.csv")
        save_table(summary, f"state_{slug(scope)}_region_extent.csv")
        note(stage, "available", detail=f"region windows={int(flagged['in_region'].sum())}")
        region_rows.append({"type": "State", "scope": scope, "status": "ok", "threshold": threshold["threshold"]})

    # Gain is computed scope-by-scope on the common physical window IDs.
    for scope in SCOPES:
        stage = f"gain:{scope}"
        raw_table = window_tables.get(("raw", scope))
        svc_table = window_tables.get(("svc", scope))
        if raw_table is None or svc_table is None:
            note(stage, "unavailable", "both Raw and SVC scope windows are required")
            continue
        raw_indexed = raw_table.set_index("window_id")
        svc_indexed = svc_table.set_index("window_id")
        common = raw_indexed.index.intersection(svc_indexed.index)
        valid = (
            raw_indexed.loc[common, "valid_window"].astype(bool)
            & svc_indexed.loc[common, "valid_window"].astype(bool)
        )
        if not valid.any():
            note(stage, "unavailable", "no common valid spatial windows")
            continue
        gain = pd.DataFrame({
            "window_id": common,
            "window_x": raw_indexed.loc[common, "window_x"].to_numpy(),
            "window_y": raw_indexed.loc[common, "window_y"].to_numpy(),
            "n_units": np.minimum(
                raw_indexed.loc[common, "n_units"].to_numpy(),
                svc_indexed.loc[common, "n_units"].to_numpy(),
            ),
            "raw_neff": raw_indexed.loc[common, "neff"].to_numpy(),
            "svc_neff": svc_indexed.loc[common, "neff"].to_numpy(),
            "valid_window": valid.to_numpy(),
        })
        gain["gain_neff"] = gain["svc_neff"] - gain["raw_neff"]
        gain = gain.loc[gain["valid_window"]].reset_index(drop=True)
        gain_tables[scope] = gain
        save_table(gain, f"gain_{slug(scope)}_common_valid_windows.csv")
        positive = gain.loc[gain["gain_neff"] > 0, "gain_neff"]
        threshold, audit = select_region_threshold(
            positive,
            n_bootstrap=REGION_N_BOOTSTRAP,
            random_state=WINDOW_SEEDS["svc"],
        )
        gain_thresholds[scope] = threshold
        save_table(audit, f"gain_{slug(scope)}_threshold_bootstrap.csv") if not audit.empty else None
        (TABLE_DIR / f"gain_{slug(scope)}_threshold.json").write_text(
            json.dumps(threshold, indent=2, default=float) + "\n",
            encoding="utf-8",
        )
        if threshold.get("threshold") is None:
            gain_region_tables[scope] = pd.DataFrame()
            note(stage, "unavailable", f"threshold status={threshold.get('status')}")
            region_rows.append({"type": "Gain", "scope": scope, "status": threshold.get("status"), "threshold": np.nan})
            continue
        summary, flagged = flag_region_windows(
            gain,
            threshold=float(threshold["threshold"]),
            value_column="gain_neff",
            window_side_length=WINDOW_SIDE_MICRONS,
        )
        gain_region_tables[scope] = flagged
        save_table(flagged, f"gain_{slug(scope)}_region_windows.csv")
        save_table(summary, f"gain_{slug(scope)}_region_extent.csv")
        note(stage, "available", detail=f"region windows={int(flagged['in_region'].sum())}")
        region_rows.append({"type": "Gain", "scope": scope, "status": "ok", "threshold": threshold["threshold"]})

region_summary = pd.DataFrame(region_rows)
if not region_summary.empty:
    save_table(region_summary, "region_summary.csv")
    display(region_summary)

if plt is not None:
    for scope, table in state_region_tables.items():
        if table.empty:
            continue
        fig, ax = plt.subplots(figsize=(7, 5))
        plot = ax.scatter(table["window_x"], table["window_y"], c=table["neff"], s=70, cmap="viridis")
        fig.colorbar(plot, ax=ax, label="SVC N_eff")
        selected = table["in_region"].astype(bool)
        ax.scatter(table.loc[selected, "window_x"], table.loc[selected, "window_y"], facecolors="none", edgecolors="black", s=150, label="State region")
        ax.set_title(f"SVC State: {scope}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"state_{slug(scope)}.png", dpi=150)
        plt.show()

    for scope, table in gain_tables.items():
        if table.empty:
            continue
        fig, ax = plt.subplots(figsize=(7, 5))
        plot = ax.scatter(table["window_x"], table["window_y"], c=table["gain_neff"], s=70, cmap="coolwarm")
        fig.colorbar(plot, ax=ax, label="N_eff(SVC) - N_eff(Raw)")
        ax.set_title(f"Window Gain: {scope}")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"gain_{slug(scope)}.png", dpi=150)
        plt.show()

## Native-coordinate Moran I

Moran I is calculated separately for the full Raw and full SVC objects. Each
side builds a deterministic six-neighbor graph from its own coordinates and
normalizes a working copy before calculating the per-gene statistic. Shared
genes, if compared later, do not imply a shared graph. Constant genes,
insufficient support, and missing coordinates remain explicit statuses.

In [ ]:
moran_tables = {}

if sample is None:
    note("moran", "unavailable", "sample input is unavailable")
else:
    for side in ("raw", "svc"):
        stage = f"moran:{side}"
        try:
            table = moran_workflow.compute_moran(
                getattr(sample, side),
                spatial_key=sample.spatial_key,
                n_neighbors=MORAN_N_NEIGHBORS,
            )
            moran_tables[side] = table
            save_table(table, f"moran_{side}.csv")
            note(stage, "available", detail=f"{len(table)} genes on native {side} graph")
            display(Markdown(f"**{side.upper()} Moran table**"))
            display(table.head(8))
        except Exception as exc:
            note(stage, "unavailable", f"{type(exc).__name__}: {exc}")

if plt is not None and moran_tables:
    fig, axes = plt.subplots(1, len(moran_tables), figsize=(6 * len(moran_tables), 4), squeeze=False)
    for axis, (side, table) in zip(axes.ravel(), moran_tables.items()):
        values = pd.to_numeric(table.loc[table["status"] == "computed", "moran"], errors="coerce").dropna()
        axis.hist(values, bins=30, color="#72B7B2", alpha=0.85)
        axis.set_title(f"{side.upper()} native Moran I")
        axis.set_xlabel("Moran I")
        axis.set_ylabel("genes")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "moran_native_distributions.png", dpi=150)
    plt.show()

## Optional AUCell pathway activity

AUCell is a side-specific score. This notebook uses the explicit synthetic
DEMO_PROGRAM G0..G49 by default so the example has a visible gene-set
definition. Set GENESET_PATH and optionally GENESET_NAMES to replace it
with a declared GMT selection, then record its coverage. The
pathway_auc_threshold is passed to the provider as its detection quantile;
it is not a post hoc score cutoff. The optional OmicVerse provider may be unavailable; that
condition is recorded for the pathway stage and does not invalidate the
partition, anatomy, window, State/Gain, or Moran stages.

In [ ]:
pathway_scores = {}
pathway_metadata = {}

if sample is None:
    note("AUCell", "unavailable", "sample input is unavailable")
elif GENE_SET_ERROR:
    note("AUCell", "unavailable", f"gene-set resource: {GENE_SET_ERROR}")
elif not GENE_SETS:
    note("AUCell", "unavailable", "no gene sets were selected")
else:
    for side, (adata, _) in side_inputs.items():
        for name, genes in GENE_SETS.items():
            stage = f"AUCell:{side}:{name}"
            try:
                result = pathway_workflow.compute_pathway_scores(
                    adata,
                    genes=list(genes),
                    score_name=name,
                    auc_threshold=PATHWAY_AUC_THRESHOLD,
                    seed=WINDOW_SEEDS[side],
                )
                score = result["scores"].rename(name)
                pathway_scores[(side, name)] = score
                pathway_metadata[(side, name)] = {
                    "coverage": result["coverage"],
                    "metadata": result["metadata"],
                }
                save_table(score, f"pathway_{side}_{slug(name)}.csv")
                (TABLE_DIR / f"pathway_{side}_{slug(name)}_metadata.json").write_text(
                    json.dumps(pathway_metadata[(side, name)], indent=2, default=float) + "\n",
                    encoding="utf-8",
                )
                note(stage, "available", detail=f"present genes={result['coverage']['n_present']}")
                display(pd.DataFrame([{
                    "side": side,
                    "program": name,
                    "n_requested": result["coverage"]["n_requested"],
                    "n_present": result["coverage"]["n_present"],
                    "fraction_present": result["coverage"]["fraction_present"],
                    "score_key": result["metadata"]["score_key"],
                }]))
            except (ImportError, RuntimeError, ValueError, KeyError) as exc:
                note(stage, "unavailable", f"{type(exc).__name__}: {exc}")

if plt is not None and pathway_scores:
    fig, axes = plt.subplots(1, len(pathway_scores), figsize=(6 * len(pathway_scores), 4), squeeze=False)
    for axis, ((side, name), scores) in zip(axes.ravel(), pathway_scores.items()):
        axis.hist(scores.to_numpy(dtype=float), bins=25, color="#ECA82C", alpha=0.85)
        axis.set_title(f"{side.upper()} {name} AUCell")
        axis.set_xlabel("AUCell score")
        axis.set_ylabel("units")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "pathway_scores.png", dpi=150)
    plt.show()

## Integrated observations and boundaries

The table below is a reading aid assembled from the stage results above. An
unavailable stage keeps its reason. Technical availability does not establish
biological validity, and a State region, Gain value, partition change, Moran
difference, or AUCell score is not by itself a claim that reconstruction
improves biology. Read the declared scale, denominators, native graphs,
support thresholds, and coverage before making a scientific conclusion.

In [ ]:
observation_rows = []

for stage, record in stage_status.items():
    observation_rows.append({
        "stage": stage,
        "status": record.get("status", ""),
        "reason": record.get("reason", ""),
        "detail": record.get("detail", ""),
    })

if not partition_summary.empty:
    observation_rows.append({
        "stage": "partitions:summary",
        "status": "available",
        "reason": "",
        "detail": f"{len(partition_summary)} independent side/scope summaries",
    })

if RUN_MEMBERSHIP and membership_summary.empty:
    observation_rows.append({
        "stage": "membership:summary",
        "status": "unavailable",
        "reason": "no Raw-defined membership comparison completed",
        "detail": "",
    })
elif not RUN_MEMBERSHIP:
    observation_rows.append({
        "stage": "membership:summary",
        "status": "disabled",
        "reason": "RUN_MEMBERSHIP is false",
        "detail": "",
    })

if anatomy_context.empty and sample is not None:
    observation_rows.append({
        "stage": "anatomy:summary",
        "status": "unavailable",
        "reason": "full Raw Level1 anatomy table is empty",
        "detail": "",
    })
elif not anatomy_context.empty:
    observation_rows.append({
        "stage": "anatomy:summary",
        "status": "available",
        "reason": "",
        "detail": f"{len(anatomy_context)} full Raw units retained",
    })

if sample is not None and level2_available:
    observation_rows.append({
        "stage": "Level2:context",
        "status": "available",
        "reason": "",
        "detail": "reported only; no anatomy inference uses Level2",
    })
elif sample is not None:
    observation_rows.append({
        "stage": "Level2:context",
        "status": "unavailable",
        "reason": f"{sample.subtype_key!r} is absent on Raw",
        "detail": "Level1 anatomy and other independent stages continue",
    })

observations = pd.DataFrame(observation_rows)
if not observations.empty:
    save_table(observations, "observations.csv")
    display(observations)

if Markdown is not None:
    display(Markdown(
        "The notebook ends with observations, not a forced success label. "
        "The batch runner remains responsible for formal result publication and "
        "the static report reads saved artifacts without opening H5AD files."
    ))